In [1]:
import numpy as np
import pandas as pd

In [2]:
restaurant = pd.read_csv("Restaurant.csv")
restaurant

,Order ID,Date,Product,Price,Quantity,Purchase Type,Payment Method,Manager,City
0,10452,07-11-2022,Fries,3.49,573.07,Online,Gift Card,Tom Jackson,London
1,10453,07-11-2022,Beverages,2.95,745.76,Online,Gift Card,Pablo Perez,Madrid
2,10454,07-11-2022,Sides & Other,4.99,200.40,In-store,Gift Card,Joao Silva,Lisbon
3,10455,08-11-2022,Burgers,12.99,569.67,In-store,Credit Card,Walter Muller,Berlin
4,10456,08-11-2022,Chicken Sandwiches,9.95,201.01,In-store,Credit Card,Walter Muller,Berlin
...,...,...,...,...,...,...,...,...,...
249,10709,28-12-2022,Sides & Other,4.99,200.40,Drive-thru,Gift Card,Walter Muller,Berlin
250,10710,29-12-2022,Burgers,12.99,754.43,Drive-thru,Gift Card,Walter Muller,Berlin
251,10711,29-12-2022,Chicken Sandwiches,9.95,281.41,Drive-thru,Gift Card,Walter Muller,Berlin
252,10712,29-12-2022,Fries,3.49,630.37,Drive-thru,Gift Card,Walter Muller,Berlin


In [3]:
#ข้อมูลใน csv
restaurant.info()                                      # to show some basic informations about the dataset

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 254 entries, 0 to 253
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Order ID        254 non-null    int64  
 1   Date            254 non-null    object 
 2   Product         254 non-null    object 
 3   Price           254 non-null    float64
 4   Quantity        254 non-null    float64
 5   Purchase Type   254 non-null    object 
 6   Payment Method  254 non-null    object 
 7   Manager         254 non-null    object 
 8   City            254 non-null    object 
dtypes: float64(2), int64(1), object(6)
memory usage: 18.0+ KB


In [4]:
#ตรวจสอบ missing values,หาค่าสถิติพื้นฐาน
restaurant[restaurant.isnull().any(axis=1)].head()
#restaurant.describe()

,Order ID,Date,Product,Price,Quantity,Purchase Type,Payment Method,Manager,City


In [5]:
#ตรวจหาข้อมูลซ้ำ
restaurant.duplicated().sum()
restaurant = restaurant.drop_duplicates()

In [6]:
#check ชนิดข้อมูล
restaurant.dtypes

Order ID            int64
Date               object
Product            object
Price             float64
Quantity          float64
Purchase Type      object
Payment Method     object
Manager            object
City               object
dtype: object

In [7]:
# ใช้ Regex เพื่อยุบช่องว่างที่เกินกว่า 1 ช่องให้เหลือช่องว่างเดียว ทำความสะอาดข้อมูล
text_columns = ['Product', 'Purchase Type', 'Payment Method', 'Manager', 'City']
for col in text_columns:
    restaurant[col] = restaurant[col].str.replace(r'\s+', ' ', regex=True).str.strip()
# ตรวจสอบความถูกต้อง
print(restaurant['Manager'].unique())

['Tom Jackson' 'Pablo Perez' 'Joao Silva' 'Walter Muller' 'Remy Monet']


In [8]:
# 5. ค่าที่เป็นไปไม่ได้ (Impossible Values) เช่น ราคาหรือจำนวนติดลบ/ศูนย์
print("\n--- 5. Impossible Values ---")
restaurant = restaurant[(restaurant['Price'] > 0) & (restaurant['Quantity'] > 0)]


--- 5. Impossible Values ---


In [9]:
Q1 = restaurant["Price"].quantile(0.25)
Q3 = restaurant["Price"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = restaurant[
    (restaurant["Price"] < lower) |
    (restaurant["Price"] > upper)
]

In [10]:
# จัดการ Outlier ด้วยวิธี Capping จำกัดค่าให้อยู่ในขอบเขต IQR ที่คำนวณไว้ ไม่ให้หลุดไปค่าที่สูงเกินจริง
restaurant['Price'] = np.where(
    restaurant['Price'] < lower, lower,
    np.where(restaurant['Price'] > upper, upper, restaurant['Price'])
)

In [11]:
print("\n--- 3. Wrong Data Type ---")
restaurant['Price'] = pd.to_numeric(restaurant['Price'], errors='coerce')
restaurant['Quantity'] = pd.to_numeric(restaurant['Quantity'], errors='coerce')


--- 3. Wrong Data Type ---


In [12]:
# 7. หน่วยไม่เหมือนกัน / ตรวจสอบความถูกต้องของค่าในคอลัมน์ข้อความ
text_cols = ['Product', 'Purchase Type', 'Payment Method', 'Manager', 'City']
print("\n--- 7. Inconsistent Units / Unique Values Check ---")
for col in text_cols:
    print(f"ค่า Unique ของ [{col}]:", restaurant[col].unique())


--- 7. Inconsistent Units / Unique Values Check ---
ค่า Unique ของ [Product]: ['Fries' 'Beverages' 'Sides & Other' 'Burgers' 'Chicken Sandwiches']
ค่า Unique ของ [Purchase Type]: ['Online' 'In-store' 'Drive-thru']
ค่า Unique ของ [Payment Method]: ['Gift Card' 'Credit Card' 'Cash']
ค่า Unique ของ [Manager]: ['Tom Jackson' 'Pablo Perez' 'Joao Silva' 'Walter Muller' 'Remy Monet']
ค่า Unique ของ [City]: ['London' 'Madrid' 'Lisbon' 'Berlin' 'Paris']


In [13]:
# Convert Date column to datetime
restaurant['Date'] = pd.to_datetime(restaurant['Date'], format='%d-%m-%Y')
# 3. ชนิดข้อมูลผิด (Wrong Data Type)

# Create additional date features
restaurant['Year'] = restaurant['Date'].dt.year
restaurant['Month'] = restaurant['Date'].dt.month
restaurant['Day'] = restaurant['Date'].dt.day
restaurant['DayOfWeek'] = restaurant['Date'].dt.dayofweek  # 0=Monday, 6=Sunday
restaurant['WeekOfYear'] = restaurant['Date'].dt.isocalendar().week

# Calculate total revenue per order
restaurant['Total_Revenue'] = restaurant['Price'] * restaurant['Quantity']

#แสดงผล
print("=== DATE RANGE ===")
print(f"From: {restaurant['Date'].min()} to {restaurant['Date'].max()}")
print(f"Total days: {(restaurant['Date'].max() - restaurant['Date'].min()).days + 1}")

=== DATE RANGE ===
From: 2022-11-07 00:00:00 to 2022-12-29 00:00:00
Total days: 53


In [14]:
outliers

,Order ID,Date,Product,Price,Quantity,Purchase Type,Payment Method,Manager,City
28,10482,13-11-2022,Fries,25.50,630.37,In-store,Credit Card,Joao Silva,Lisbon
29,10486,14-11-2022,Chicken Sandwiches,29.05,201.01,In-store,Credit Card,Joao Silva,Lisbon


In [15]:
# แปลงข้อความทุกคอลัมน์ให้เป็นตัวพิมพ์เล็กทั้งหมด ป้องกันปัญหาชื่อหมวดหมู่ซ้ำซ้อน
text_columns = ['Product', 'Purchase Type', 'Payment Method', 'Manager', 'City']
for col in text_columns:
    restaurant[col] = restaurant[col].apply(lambda x: x.lower() if isinstance(x, str) else x) #แทนการใช้ .str.lower() ตรงๆ วิธีนี้จะช่วยป้องกัน 
#Error กรณีที่ข้อมูลในคอลัมน์มีค่าว่าง (NaN) หรือมีตัวเลขปะปนอยู่ โดยจะแปลงเฉพาะข้อมูลที่เป็นข้อความจริง ๆ เท่านั้น และคงค่า NaN ไว้ตามเดิม

In [16]:
#ปรับค่าให้เขียนตรงกันเช็คค่าที่ไม่ซ้ำกัน
print(restaurant["Product"].unique())
print(restaurant["Price"].unique())
print(restaurant["Quantity"].unique())
print(restaurant["Purchase Type"].unique())
print(restaurant["Payment Method"].unique())
print(restaurant["Manager"].unique())
print(restaurant["City"].unique())

['fries' 'beverages' 'sides & other' 'burgers' 'chicken sandwiches']
[ 3.49  2.95  4.99 12.99  9.95 19.64]
[573.07 745.76 200.4  569.67 201.01 554.27 677.97 630.37 523.48 508.08
 538.88 687.68 477.29 492.69 461.89 446.5  585.07 221.11 600.46 631.25
 646.65 677.44 241.21 261.31 692.84 281.41 723.63 301.51 754.43]
['online' 'in-store' 'drive-thru']
['gift card' 'credit card' 'cash']
['tom jackson' 'pablo perez' 'joao silva' 'walter muller' 'remy monet']
['london' 'madrid' 'lisbon' 'berlin' 'paris']


In [17]:
#ลบช่องว่างส่วนเกิน

restaurant["Product"] = restaurant["Product"].str.strip()
restaurant["Purchase Type"] = restaurant["Purchase Type"].str.strip()
restaurant["Payment Method"] = restaurant["Payment Method"].str.strip()
restaurant["Manager"] = restaurant["Manager"].str.strip()
restaurant["City"] = restaurant["City"].str.strip()

In [18]:
#นับจำนวนข้อมูลของแต่ละคอลัม
print(restaurant["Product"].value_counts())
print(restaurant["Price"].value_counts())
print(restaurant["Quantity"].value_counts())
print(restaurant["Purchase Type"].value_counts())
print(restaurant["Payment Method"].value_counts())
print(restaurant["Manager"].value_counts())
print(restaurant["City"].value_counts())

Product
burgers               52
chicken sandwiches    52
fries                 51
beverages             50
sides & other         49
Name: count, dtype: int64
Price
12.99    52
9.95     51
3.49     50
2.95     50
4.99     49
19.64     2
Name: count, dtype: int64
Quantity
200.40    49
201.01    36
630.37    35
677.97    34
745.76    16
573.07     9
221.11     8
687.68     7
523.48     6
569.67     6
538.88     5
554.27     5
477.29     5
508.08     4
677.44     4
281.41     3
241.21     3
461.89     3
492.69     3
646.65     2
585.07     2
692.84     2
600.46     1
631.25     1
446.50     1
261.31     1
723.63     1
301.51     1
754.43     1
Name: count, dtype: int64
Purchase Type
online        107
in-store       86
drive-thru     61
Name: count, dtype: int64
Payment Method
credit card    120
cash            76
gift card       58
Name: count, dtype: int64
Manager
tom jackson      75
joao silva       75
pablo perez      46
walter muller    30
remy monet       28
Name: count, dtype: int64

In [19]:
# โหลดข้อมูลใหม่และแปลง One-Hot Encoding แบบรวบยอด
restaurant = pd.read_csv("Restaurant.csv")
restaurant["Product"] = restaurant["Product"].str.lower()

categorical_cols = ['Product', 'Purchase Type', 'Payment Method', 'Manager', 'City']
restaurant = pd.get_dummies(restaurant, columns=categorical_cols, drop_first=True)

In [20]:
# ตรวจสอบความสะอาดรอบสุดท้าย (เช็คว่าไม่มีค่า Null หหลงเหลือ)
assert restaurant.isnull().sum().sum() == 0, 'Have Missing Values!'

# ส่งออกชุดข้อมูลที่คลีนแล้วเป็นไฟล์ CSV
restaurant.to_csv('cleaned_Restaurant.csv', index=False)
print("save file cleaned_re.csv success")

save file cleaned_re.csv success


In [22]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# โหลดข้อมูลดิบตั้งต้นใหม่เพื่อป้องกันข้อมูลถูกแปลงทับ แก้ปัญหา error
restaurant = pd.read_csv("Restaurant.csv")

# คัดลอก DataFrame สำหรับทำโมเดล
restaurant_model = restaurant.copy()

# One-Hot Encoding
categorical_cols = ['Product', 'Purchase Type', 'Payment Method', 'Manager', 'City']
restaurant_model = pd.get_dummies(restaurant_model, columns=categorical_cols, drop_first=True)

# Standardize ตัวเลข (Price, Quantity) 
scaler = StandardScaler()
numeric_cols = ['Price', 'Quantity']
restaurant_model[numeric_cols] = scaler.fit_transform(restaurant_model[numeric_cols])

print(restaurant_model.head())

   Order ID        Date     Price  Quantity  Product_Burgers  \
0     10452  07-11-2022 -0.833620  0.524367            False   
1     10453  07-11-2022 -0.958236  1.329579            False   
2     10454  07-11-2022 -0.487463 -1.213303            False   
3     10455  08-11-2022  1.358705  0.508514             True   
4     10456  08-11-2022  0.657161 -1.210459            False   

   Product_Chicken Sandwiches  Product_Fries  Product_Sides & Other  \
0                       False           True                  False   
1                       False          False                  False   
2                       False          False                   True   
3                       False          False                  False   
4                        True          False                  False   

   Purchase Type_In-store   Purchase Type_Online   ...  \
0                    False                   True  ...   
1                    False                   True  ...   
2             